# Notebook 1: Pre-Ingestion File Validation

**Purpose:** Validate JSON file before Auto Loader ingestion

**Checks:**
1. File exists and has valid size
2. File extension is .json
3. Content is actually JSON (not CSV/XML/text)
4. JSON can be parsed by Spark
5. Corrupt/malformed record detection
6. Schema structure
7. Array fields and explosion factors

**Output:** GO/CAUTION/STOP decision + Auto Loader config recommendations


## Configuration


In [0]:
# ===== MODIFY THIS =====
FILE_PATH = "/Volumes/dev_automotive/landing/landing_raw/catalogs.json"

# Or use a widget (uncomment to use):
# dbutils.widgets.text("file_path", "/Volumes/dev_automotive/landing/landing_raw/catalogs.json", "JSON file path")
# FILE_PATH = dbutils.widgets.get("file_path").strip()


## Validation Start


In [0]:
from pyspark.sql import functions as F

# Lean: one line enough; banner below is optional
print(f"File: {FILE_PATH}")
# print("="*70)
# print("FILE VALIDATION")
# print("="*70)
# print()


## Step 1: File System Check


In [0]:
# Step 1: exist + non-empty. Extension check is optional (Step 3 will catch wrong format).
try:
    file_info = dbutils.fs.ls(FILE_PATH)[0]
    file_size_mb = round(file_info.size / (1024 * 1024), 2)
    file_name = file_info.name
    # Optional: warn only, don't block
    # if not file_name.lower().endswith('.json'):
    #     print(f"   ⚠️  Extension not .json: {file_name}")
    if file_size_mb == 0:
        print(f"   ❌ File is EMPTY (0 MB)")
        dbutils.notebook.exit("STOP: Empty file")
    else:
        print(f"   ✅ File size: {file_size_mb} MB")
except Exception as e:
    print(f"   ❌ File NOT found: {e}")
    dbutils.notebook.exit("STOP: File not found")


## Step 2: JSON Parsing + Corrupt Record Detection


In [0]:
# Step 3: core — parse + corrupt count. Rest is optional.
try:
    df = spark.read \
        .option("mode", "PERMISSIVE") \
        .option("columnNameOfCorruptRecord", "_corrupt_record") \
        .json(FILE_PATH)
    
    total_count = df.count()
    corrupt_count = 0
    if "_corrupt_record" in df.columns:
        corrupt_count = df.filter(F.col("_corrupt_record").isNotNull()).count()
    
    clean_count = total_count - corrupt_count
    
    print(f"   ✅ Successfully parsed as JSON")
    print(f"   ✅ Total records: {total_count:,}")
    print(f"   ✅ Clean records: {clean_count:,}")
    
    if corrupt_count > 0:
        corrupt_pct = round(corrupt_count/total_count*100, 1)
        print(f"   ⚠️  Corrupt: {corrupt_count:,} ({corrupt_pct}%) — enable badRecordsPath in Auto Loader")
        # Optional: show sample corrupt line
        # corrupt_sample = df.filter(F.col("_corrupt_record").isNotNull()).select("_corrupt_record").limit(1).collect()
        # if corrupt_sample: print(f"   Sample: {str(corrupt_sample[0][0])[:200]}...")
    else:
        print(f"   ✅ No corrupt records")
    
    if total_count == 0:
        print(f"   ❌ No records in file!")
        dbutils.notebook.exit("STOP: Empty JSON")
        
except Exception as e:
    print(f"   ❌ CANNOT parse as JSON")
    print(f"   Error: {str(e)[:200]}...")
    print(f"   → Verify file format at source")
    dbutils.notebook.exit("STOP: Cannot parse as JSON")

print()


## Step 3: Schema Analysis


In [0]:
df.printSchema()


## Step 4: Array Detection (Explosion Factor)


In [0]:
print("5️⃣ Array check...")

array_fields = [f.name for f in df.schema.fields 
                if "array" in str(f.dataType).lower() 
                and f.name != "_corrupt_record"]

if array_fields:
    print(f"   ⚠️  Arrays found: {', '.join(array_fields)}")
    print()
    for arr_col in array_fields:
        try:
            avg_len = df.select(F.avg(F.size(arr_col))).first()[0]
            if avg_len:
                exploded_count = int(clean_count * avg_len)
                print(f"   Array: '{arr_col}'")
                print(f"   ├─ Avg length: {round(avg_len, 1)}")
                print(f"   ├─ Current rows: {clean_count:,}")
                print(f"   └─ After explosion: ~{exploded_count:,} rows ({round(avg_len, 1)}x)")
                print()
        except Exception as e:
            print(f"   ⚠️  Could not analyze array '{arr_col}': {e}")
    print(f"   💡 Recommendation: DON'T explode in Bronze")
    print(f"      → Keep nested structure intact")
    print(f"      → Do explosion in Silver layer with proper business logic")
else:
    print(f"   ✅ No arrays - flat structure")

print()


## Step 5: Sample Data Preview


In [0]:
display(df.limit(5))


## Step 6: Final Decision


In [0]:
issues = []
warnings = []
if corrupt_count > 0:
    pct = round(corrupt_count / total_count * 100, 1)
    (issues if pct > 10 else warnings).append(f"Corrupt: {corrupt_count:,} ({pct}%)")
# if file_size_mb > 1000: warnings.append(f"Large file: {file_size_mb} MB")  # optional
# if not file_name.lower().endswith('.json'): warnings.append(f"Extension: {file_name}")

if clean_count == 0:
    print("❌ STOP - No valid records")
    dbutils.notebook.exit("STOP: No valid records")

if issues:
    print("⚠️  GO WITH CAUTION"); [print(f"   🔴 {i}") for i in issues]
elif warnings:
    print("⚠️  GO"); [print(f"   ⚠️  {w}") for w in warnings]
else:
    print("✅ GO - File is healthy")

print()
print("Next: Notebook 2 (Auto Loader). Use cloudFiles.format=json, schemaLocation, rescue mode; badRecordsPath if corrupt.")
if array_fields:
    print(f"Arrays: keep intact in Bronze — {', '.join(array_fields)}")


## Summary

**Next Steps:**
- If decision = **GO** → Proceed to Notebook 2 (Auto Loader)
- If decision = **CAUTION** → Review warnings, then proceed
- If decision = **STOP** → Fix issues before ingestion

